In [1]:
import jax
import flax
import optax
from jax import lax, value_and_grad, numpy as jnp
from jax import random, vmap, jacfwd, jit
from jax import config
from flax import linen as nn
import jax_dataloader as jd

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import xarray as xr
from torch.utils.data import Dataset
import os
from tqdm.auto import tqdm

# Optimize JAX for High-Performance VRAM Allocation
config.update("jax_enable_x64", True)
os.environ['XLA_PYTHON_CLIENT_ALLOCATOR'] = 'cuda_async'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'true'

In [2]:
DATA_DIR = Path.cwd().parent.parent / "data"
POISSON_FILE = DATA_DIR / "poisson-gauss" / "snapshots" / "Poisson-Gauss.nc"
HELMHOLTZ_FILE = DATA_DIR / "Helmholtz-Sinusoidal.nc"

class PoissonGaussDataset(Dataset):
    def __init__(self, source_data, solution_data):
        self.s = source_data.reshape(source_data.shape[0], -1, 1)
        self.u = solution_data.reshape(solution_data.shape[0], -1, 1)

    def __len__(self): return len(self.s)
    def __getitem__(self, idx): return self.s[idx], self.u[idx]

class HelmholtzDataset(Dataset):
    def __init__(self, source_data, solution_data, k_data):
        self.q = source_data.reshape(source_data.shape[0], -1, 1)
        self.u = solution_data.reshape(solution_data.shape[0], -1, 1)
        self.k = k_data.reshape(-1, 1) 

    def __len__(self): return len(self.u)
    def __getitem__(self, idx): return self.q[idx], self.k[idx], self.u[idx]

# Restrict to first 1k samples
n_total = 1000
n_train, n_val, n_test = 800, 100, 100

# ---- Load Poisson ----
with xr.open_dataset(POISSON_FILE) as ds_p:
    p_src = ds_p['source'].values[:n_total]
    p_sol = ds_p['solution'].values[:n_total]

train_ds_p = PoissonGaussDataset(p_src[:n_train], p_sol[:n_train])
val_ds_p = PoissonGaussDataset(p_src[n_train:n_train+n_val], p_sol[n_train:n_train+n_val])
test_ds_p = PoissonGaussDataset(p_src[-n_test:], p_sol[-n_test:])

# ---- Load Helmholtz ----
with xr.open_dataset(HELMHOLTZ_FILE) as ds_h:
    h_src = ds_h['source'].values[:n_total]
    h_sol = ds_h['solution'].values[:n_total]
    h_k = ds_h['k'].values[:n_total]

train_ds_h = HelmholtzDataset(h_src[:n_train], h_sol[:n_train], h_k[:n_train])
val_ds_h = HelmholtzDataset(h_src[n_train:n_train+n_val], h_sol[n_train:n_train+n_val], h_k[n_train:n_train+n_val])
test_ds_h = HelmholtzDataset(h_src[-n_test:], h_sol[-n_test:], h_k[-n_test:])

batch_size = 8
train_dl_p = jd.DataLoader(train_ds_p, backend='pytorch', batch_size=batch_size, shuffle=True, drop_last=True)
train_dl_h = jd.DataLoader(train_ds_h, backend='pytorch', batch_size=batch_size, shuffle=True, drop_last=True)

val_dl_p = jd.DataLoader(val_ds_p, backend='pytorch', batch_size=batch_size, shuffle=False)
val_dl_h = jd.DataLoader(val_ds_h, backend='pytorch', batch_size=batch_size, shuffle=False)

print("DataLoaders initialized: Combined Foundation Training.")

DataLoaders initialized: Combined Foundation Training.


In [3]:
nx, ny = 128, 128
x_coords = np.linspace(0.0, 1.0, nx, dtype=np.float64)
y_coords = np.linspace(0.0, 1.0, ny, dtype=np.float64)
X, Y = np.meshgrid(x_coords, y_coords, indexing='ij')

x = jnp.array(X.reshape(-1, 1))
y = jnp.array(Y.reshape(-1, 1))

# Dirichlet Boundary Condition Masking (used by both datasets)
bc_left, bc_right = (x == 0.0).flatten(), (x == 1.0).flatten()
bc_bottom, bc_top = (y == 0.0).flatten(), (y == 1.0).flatten()
bc_all = bc_left | bc_right | bc_bottom | bc_top 

# Extract coordinates
x_bc, y_bc = x[bc_all], y[bc_all]
interior_mask = ~bc_all
x_interior, y_interior = x[interior_mask], y[interior_mask]

print(f"PDE Interior Grid Nodes: {x_interior.shape[0]} | BC Nodes: {x_bc.shape[0]}")

E0727 23:42:39.840009   62726 cuda_executor.cc:1526] Could not get kernel mode driver version: (INVALID_ARGUMENT: Version does not match the format X.Y.Z)
E0727 23:42:39.910440   62587 cuda_executor.cc:1526] Could not get kernel mode driver version: (INVALID_ARGUMENT: Version does not match the format X.Y.Z)


PDE Interior Grid Nodes: 15876 | BC Nodes: 508


In [4]:
epochs = 5
M = 512 
chunk_size = 256 
hidden_layers = [256, 256, 256, 256] 
# total_features = hidden_layers[-1] # for no concat
total_features = sum(hidden_layers)

# Immutable Base Hyperparameters
sigma = 1.0 
tik_reg_fixed = 1e-5 
matrix_bc_weight = 1e4
pde_loss_weight = 1.0
bc_loss_weight = 1e4

# Fixed B Matrix projection mapping
rff_key = jax.random.PRNGKey(99)
B_matrix = jax.random.normal(rff_key, (2, M)) * sigma

# class FeatureExtractor(nn.Module):
#     hidden_layers: list
#     B_matrix: jnp.ndarray 
    
#     @nn.compact
#     def __call__(self, x_in, y_in):    
#         v = jnp.concatenate([x_in, y_in])
#         proj = 2.0 * jnp.pi * jnp.dot(v, self.B_matrix)
#         h = jnp.concatenate([jnp.sin(proj), jnp.cos(proj)])

#         for size in self.hidden_layers:
#             h = nn.Dense(size, kernel_init=nn.initializers.he_normal())(h)
#             h = nn.tanh(h) 
            
#         return h

class FeatureExtractor(nn.Module):
    hidden_layers: list
    B_matrix: jnp.ndarray 
    
    @nn.compact
    def __call__(self, x_in, y_in):    
        v = jnp.concatenate([x_in, y_in])
        proj = 2.0 * jnp.pi * jnp.dot(v, self.B_matrix)
        h_rff = jnp.concatenate([jnp.sin(proj), jnp.cos(proj)])
        all_features = []

        h = h_rff
        for size in self.hidden_layers:
            h = nn.Dense(size, kernel_init=nn.initializers.he_normal())(h)
            h = nn.tanh(h) 
            all_features.append(h)
        
        combined_h = jnp.concatenate(all_features)
            
        return combined_h 

model = FeatureExtractor(hidden_layers=hidden_layers, B_matrix=B_matrix)
params = model.init(jax.random.PRNGKey(0), jnp.array([0.0]), jnp.array([0.0]))

In [5]:
def get_f_chunk(params, x_val, y_val, start_idx, end_idx):
    f = model.apply(params, x_val, y_val)
    return lax.dynamic_slice(f, (start_idx,), (end_idx - start_idx,))

def get_f_dir_chunk(params, x_val, y_val, start_idx, end_idx):
    f_xx_chunk = jacfwd(jacfwd(get_f_chunk, argnums=1), argnums=1)(params, x_val, y_val, start_idx, end_idx)
    f_yy_chunk = jacfwd(jacfwd(get_f_chunk, argnums=2), argnums=2)(params, x_val, y_val, start_idx, end_idx)
    return jnp.squeeze(f_xx_chunk, axis=(-1, -2)), jnp.squeeze(f_yy_chunk, axis=(-1, -2))

f_spatial_vmap = vmap(lambda p, x, y: model.apply(p, x, y), in_axes=(None, 0, 0))
f_chunk_spatial_vmap = vmap(get_f_chunk, in_axes=(None, 0, 0, None, None))
f_dir_chunk_spatial_vmap = vmap(get_f_dir_chunk, in_axes=(None, 0, 0, None, None))

In [6]:
# ==========================================
# POISSON-GAUSS PHYSICS SOLVER
# Equation: nabla^2 u = -s
# ==========================================
def get_A_b_poisson(params, x_pde, y_pde, s_pde, x_bc, y_bc, total_features, chunk_size):
    A_chunks = []
    for start_idx in range(0, total_features, chunk_size):
        end_idx = min(start_idx + chunk_size, total_features)
        f_xx, f_yy = f_dir_chunk_spatial_vmap(params, x_pde, y_pde, start_idx, end_idx)
        
        A_pde_chunk = f_xx + f_yy
        f_bc_chunk = f_chunk_spatial_vmap(params, x_bc, y_bc, start_idx, end_idx)
        A_bc_chunk = f_bc_chunk * jnp.sqrt(matrix_bc_weight)
        
        A_chunks.append(jnp.vstack([A_pde_chunk, A_bc_chunk]))
        
    A_full = jnp.hstack(A_chunks)
    b_full = jnp.vstack([-s_pde, jnp.zeros_like(x_bc) * jnp.sqrt(matrix_bc_weight)])
    return A_full, b_full

def evaluate_poisson_sample(params, x_pde, y_pde, s_sample, x_bc, y_bc, u_sample):
    A, b = get_A_b_poisson(params, x_pde, y_pde, s_sample, x_bc, y_bc, total_features, chunk_size)
    AtA = A.T @ A + tik_reg_fixed * jnp.eye(total_features)
    w_curr = jax.scipy.linalg.solve(AtA, A.T @ b, assume_a='pos')
    
    u_pred_interior = jnp.dot(f_spatial_vmap(params, x_pde, y_pde), w_curr)
    u_pred_bc = jnp.dot(f_spatial_vmap(params, x_bc, y_bc), w_curr)
    
    loss_pde = jnp.mean((A[:x_pde.shape[0]] @ w_curr - b[:x_pde.shape[0]]) ** 2)
    loss_bc = jnp.mean((u_pred_bc - 0.0) ** 2)
    loss_data = jnp.mean((u_pred_interior - u_sample) ** 2)
    
    return (loss_pde * pde_loss_weight, loss_bc * bc_loss_weight), (loss_pde, loss_bc, loss_data)

# ==========================================
# HELMHOLTZ ANALYTICAL PHYSICS SOLVER
# Equation: nabla^2 u + k u = q
# ==========================================
def get_A_b_helmholtz(params, x_pde, y_pde, q_pde, k_val, x_bc, y_bc, total_features, chunk_size):
    A_chunks = []
    for start_idx in range(0, total_features, chunk_size):
        end_idx = min(start_idx + chunk_size, total_features)
        f_xx, f_yy = f_dir_chunk_spatial_vmap(params, x_pde, y_pde, start_idx, end_idx)
        f_chunk = f_chunk_spatial_vmap(params, x_pde, y_pde, start_idx, end_idx)
        
        A_pde_chunk = f_xx + f_yy + k_val * f_chunk
        f_bc_chunk = f_chunk_spatial_vmap(params, x_bc, y_bc, start_idx, end_idx)
        A_bc_chunk = f_bc_chunk * jnp.sqrt(matrix_bc_weight)
        
        A_chunks.append(jnp.vstack([A_pde_chunk, A_bc_chunk]))
        
    A_full = jnp.hstack(A_chunks)
    b_full = jnp.vstack([q_pde, jnp.zeros_like(x_bc) * jnp.sqrt(matrix_bc_weight)])
    return A_full, b_full

def evaluate_helmholtz_sample(params, x_pde, y_pde, q_sample, k_sample, x_bc, y_bc, u_sample):
    A, b = get_A_b_helmholtz(params, x_pde, y_pde, q_sample, k_sample, x_bc, y_bc, total_features, chunk_size)
    AtA = A.T @ A + tik_reg_fixed * jnp.eye(total_features)
    w_curr = jax.scipy.linalg.solve(AtA, A.T @ b, assume_a='pos')
    
    u_pred_interior = jnp.dot(f_spatial_vmap(params, x_pde, y_pde), w_curr)
    u_pred_bc = jnp.dot(f_spatial_vmap(params, x_bc, y_bc), w_curr)
    
    loss_pde = jnp.mean((A[:x_pde.shape[0]] @ w_curr - b[:x_pde.shape[0]]) ** 2)
    loss_bc = jnp.mean((u_pred_bc - 0.0) ** 2)
    loss_data = jnp.mean((u_pred_interior - u_sample) ** 2)
    
    return (loss_pde * pde_loss_weight, loss_bc * bc_loss_weight), (loss_pde, loss_bc, loss_data)

batched_poisson = jax.vmap(evaluate_poisson_sample, in_axes=(None, None, None, 0, None, None, 0))
batched_helmholtz = jax.vmap(evaluate_helmholtz_sample, in_axes=(None, None, None, 0, 0, None, None, 0))

In [7]:
optimizer = optax.adamw(learning_rate=optax.cosine_decay_schedule(init_value=1e-3, decay_steps=epochs, alpha=0.1))

def combined_loss_eval(params, x_pde, y_pde, p_s, p_u, h_q, h_k, h_u, x_bc, y_bc):
    # Compute batched losses
    (p_pde_sc, p_bc_sc), aux_p = batched_poisson(params, x_pde, y_pde, p_s, x_bc, y_bc, p_u)
    (h_pde_sc, h_bc_sc), aux_h = batched_helmholtz(params, x_pde, y_pde, h_q, h_k, x_bc, y_bc, h_u)
    
    # Calculate sum across ALL drawn samples
    sum_pde = jnp.sum(p_pde_sc) + jnp.sum(h_pde_sc)
    sum_bc = jnp.sum(p_bc_sc) + jnp.sum(h_bc_sc)
    
    # Aux unscaled metric averages (for reporting logic)
    metrics = {
        'p_pde': jnp.mean(aux_p[0]), 'p_bc': jnp.mean(aux_p[1]), 'p_data': jnp.mean(aux_p[2]),
        'h_pde': jnp.mean(aux_h[0]), 'h_bc': jnp.mean(aux_h[1]), 'h_data': jnp.mean(aux_h[2]),
        'tot_loss_val': sum_pde + sum_bc
    }
    return (sum_pde, sum_bc), metrics

@jax.jit
def update_network(params, opt_state, x_batch, y_batch, p_s, p_u, h_q, h_k, h_u, x_bc, y_bc):
    loss_tuple, vjp_fn, metrics = jax.vjp(
        lambda p: combined_loss_eval(p, x_batch, y_batch, p_s, p_u, h_q, h_k, h_u, x_bc, y_bc),
        params, has_aux=True
    )
    
    # Isolate & capture individual PDE vs BC Gradients for analysis
    pde_grads = vjp_fn((1.0, 0.0))[0]
    bc_grads = vjp_fn((0.0, 1.0))[0]
    total_grads = jax.tree_util.tree_map(lambda x, y: x + y, pde_grads, bc_grads)
    
    updates, opt_state = optimizer.update(total_grads, opt_state, params=params)
    new_params = optax.apply_updates(params, updates)
    
    grad_norms = (optax.global_norm(total_grads), optax.global_norm(pde_grads), optax.global_norm(bc_grads))
    
    return new_params, opt_state, metrics, grad_norms

In [ ]:
opt_state = optimizer.init(params)
rng = jax.random.PRNGKey(42)

spatial_batch_size = 5000
history = {k: [] for k in ['total', 'p_pde', 'p_bc', 'p_data', 'h_pde', 'h_bc', 'h_data']}

for epoch in range(epochs):
    ep_metrics = {k: 0.0 for k in history.keys()}
    batches = 0
    
    # Zip datasets together ensuring "completely unrelated" sample fetching
    train_combo = zip(train_dl_p, train_dl_h)
    
    for (p_s, p_u), (h_q, h_k, h_u) in tqdm(train_combo, total=n_train//batch_size, desc=f"Epoch {epoch+1}/{epochs}"):
        rng, key = jax.random.split(rng)
        
        # Consistent Spatial Sampling mapped across both independent formulations
        b_idx = jax.random.choice(key, x_interior.shape[0], shape=(spatial_batch_size,), replace=False)
        x_chunk, y_chunk = x_interior[b_idx], y_interior[b_idx]
        
        # Apply masks and spatial slice
        p_s_c = p_s[:, interior_mask][:, b_idx, :]
        p_u_c = p_u[:, interior_mask][:, b_idx, :]
        h_q_c = h_q[:, interior_mask][:, b_idx, :]
        h_u_c = h_u[:, interior_mask][:, b_idx, :]
        
        params, opt_state, metrics, norms = update_network(
            params, opt_state, x_chunk, y_chunk, p_s_c, p_u_c, h_q_c, h_k, h_u_c, x_bc, y_bc
        )
        
        for k in history.keys():
            ep_metrics[k] += float(metrics.get(k, metrics.get('tot_loss_val', 0.0)))
        batches += 1
        
    for k in history.keys(): history[k].append(ep_metrics[k] / batches)
    
    g_tot, g_pde, g_bc = norms
    print(f"E{epoch+1:02d} | T-Loss: {history['total'][-1]:.2e} | P-Data: {history['p_data'][-1]:.2e} | H-Data: {history['h_data'][-1]:.2e}")
    print(f"      └─ Grad Norms => Tot: {g_tot:.3e} | PDE: {g_pde:.3e} | BC: {g_bc:.3e}")

Epoch 1/5:   0%|          | 0/100 [00:00<?, ?it/s]

E01 | T-Loss: 5.82e+02 | P-Data: 2.57e-09 | H-Data: 2.37e-04
      └─ Grad Norms => Tot: 3.422e+03 | PDE: 2.876e+03 | BC: 1.694e+03


Epoch 2/5:   0%|          | 0/100 [00:00<?, ?it/s]

E02 | T-Loss: 1.02e+02 | P-Data: 2.48e-09 | H-Data: 2.95e-05
      └─ Grad Norms => Tot: 2.189e+03 | PDE: 1.712e+03 | BC: 1.201e+03


Epoch 3/5:   0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 5))

# Total Sum Loss
axes[0].plot(history['total'], color='black', linewidth=2)
axes[0].set_yscale('log')
axes[0].set_title('Overall Total Loss (Summed)', fontsize=14)
axes[0].grid(True, which="both", ls="-", alpha=0.3)

# PDE Losses
axes[1].plot(history['p_pde'], color='blue', label='Poisson PDE', linewidth=2)
axes[1].plot(history['h_pde'], color='purple', label='Helmholtz PDE', linewidth=2)
axes[1].set_yscale('log')
axes[1].legend()
axes[1].set_title('PDE Residual Losses', fontsize=14)
axes[1].grid(True, which="both", ls="-", alpha=0.3)

# BC Losses
axes[2].plot(history['p_bc'], color='green', label='Poisson BC', linewidth=2)
axes[2].plot(history['h_bc'], color='lime', label='Helmholtz BC', linewidth=2)
axes[2].set_yscale('log')
axes[2].legend()
axes[2].set_title('Boundary Condition Losses', fontsize=14)
axes[2].grid(True, which="both", ls="-", alpha=0.3)

# Data MSE
axes[3].plot(history['p_data'], color='red', label='Poisson MSE', linewidth=2)
axes[3].plot(history['h_data'], color='orange', label='Helmholtz MSE', linewidth=2)
axes[3].set_yscale('log')
axes[3].legend()
axes[3].set_title('Reporting Data MSE', fontsize=14)
axes[3].grid(True, which="both", ls="-", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
s_sample, u_sim_sample = test_ds_p[12]

# Formulate Full Dense Evaluation (No chunking needed for inference map)
s_batch = s_sample[None, ...]
A_p, b_p = get_A_b_poisson(params, x, y, s_batch[0], x_bc, y_bc, total_features, chunk_size)
AtA = A_p.T @ A_p + tik_reg_fixed * jnp.eye(total_features)
w_plot = jax.scipy.linalg.solve(AtA, A_p.T @ b_p, assume_a='pos')

f_features = f_spatial_vmap(params, x, y)
u_pred = f_features @ w_plot

s_plot = s_sample.reshape(nx, ny).T
u_sim_plot = u_sim_sample.reshape(nx, ny).T 
u_pred_plot = u_pred.reshape(nx, ny).T

ext = [0, nx - 1, 0, ny - 1]
fig = plt.figure(figsize=(18, 5))
titles = ['Test Source (s)', 'Ground Truth (u)', 'PINN Prediction']
maps = [s_plot, u_sim_plot, u_pred_plot]

for i, (m, t) in enumerate(zip(maps, titles)):
    ax = fig.add_subplot(1, 3, i+1)
    mesh = ax.imshow(m, interpolation='bilinear', origin='lower', cmap='rainbow', extent=ext, aspect='auto')
    fig.colorbar(mesh, ax=ax)
    ax.set_title(t, fontsize=14)
    ax.set_xlabel('x')
    if i == 0: ax.set_ylabel('y')

plt.tight_layout()
plt.show()

In [ ]:
q_sample, k_sample, u_sim_sample = test_ds_h[3]

q_batch = q_sample[None, ...]
k_batch = k_sample[None, ...]
A_h, b_h = get_A_b_helmholtz(params, x, y, q_batch[0], k_batch[0], x_bc, y_bc, total_features, chunk_size)
AtA = A_h.T @ A_h + tik_reg_fixed * jnp.eye(total_features)
w_plot = jax.scipy.linalg.solve(AtA, A_h.T @ b_h, assume_a='pos')

f_features = f_spatial_vmap(params, x, y)
u_pred = f_features @ w_plot

q_plot = q_sample.reshape(nx, ny).T
u_sim_plot = u_sim_sample.reshape(nx, ny).T 
u_pred_plot = u_pred.reshape(nx, ny).T

ext = [0.0, 1.0, 0.0, 1.0]
fig = plt.figure(figsize=(18, 5))
titles = [f'Test Source q(x,y) | k = {float(k_sample):.4f}', 'Test Ground Truth (u)', 'PINN Generalization Prediction']
maps = [q_plot, u_sim_plot, u_pred_plot]

for i, (m, t) in enumerate(zip(maps, titles)):
    ax = fig.add_subplot(1, 3, i+1)
    mesh = ax.imshow(m, interpolation='bilinear', origin='lower', cmap='rainbow', extent=ext, aspect='auto')
    fig.colorbar(mesh, ax=ax)
    ax.set_title(t, fontsize=14)
    ax.set_xlabel('x')
    if i == 0: ax.set_ylabel('y')

plt.tight_layout()
plt.show()